In [ ]:
from cellpose import models, io
import pyclesperanto as cle
from bioio import BioImage
import napari
from skimage.transform import rescale, resize
from skimage.io import imsave, imread
from skimage.measure import label, regionprops, regionprops_table, marching_cubes, mesh_surface_area
from scipy import ndimage
import numpy as np
import random
from glob import glob
from skimage import exposure
import pandas as pd

cle.select_device("NVIDIA")

In [ ]:
def get_pixel_spacing(img):
    ''' 
    Use one image from dataset read in with BioImage without .get_image_data to access pixel spacing
    '''
    if img.dims['Z'][0] > 1:
        pixel_spacing = (img.physical_pixel_sizes[0],img.physical_pixel_sizes[1],img.physical_pixel_sizes[2])
    else:
        pixel_spacing = (img.physical_pixel_sizes[0],img.physical_pixel_sizes[1])
    return pixel_spacing

def normalize_images(input_image,tophat_radius=10):
    input_gpu = cle.push(input_image)
    #normalizing the image stack
    equalized_intensities_stack = cle.create_like(input_gpu)
    a_slice = cle.create([input_gpu.shape[1], input_gpu.shape[2]])
    num_slices = input_gpu.shape[0]
    mean_intensity_stack = cle.mean_of_all_pixels(input_gpu)
    corrected_slice = None
    for z in range(0, num_slices):
        # get a single slice out of the stack
        cle.copy_slice(input_gpu, a_slice, z)
        # measure its intensity
        mean_intensity_slice = cle.mean_of_all_pixels(a_slice)
        # correct the intensity
        correction_factor = mean_intensity_slice/mean_intensity_stack
        corrected_slice = cle.multiply_image_and_scalar(a_slice, corrected_slice, correction_factor)
        # copy slice back in a stack
        cle.copy_slice(corrected_slice, equalized_intensities_stack, z)
    #background subtraction (increase the signal to noise ratio for improved segmentation results)
    background_subtracted_top_hat = cle.top_hat_sphere(equalized_intensities_stack,radius_x=tophat_radius,radius_y=tophat_radius,radius_z=tophat_radius)
    #pull data off gpu
    input_pull = cle.pull(input_gpu)
    background_subtracted_top_hat_pull = cle.pull(background_subtracted_top_hat)
    equalized_intensities_stack_pull = cle.pull(equalized_intensities_stack)
    return background_subtracted_top_hat_pull

def down_size(img, scale=None):
   scale == 1 if scale == None else scale == scale
   if scale == None:
       print('no scaling provided, using scaling of 1 (no scaling applied)')
   return rescale(img, scale, anti_aliasing=True, channel_axis=None,preserve_range=True)

def run_cpsam(imgs):
    '''
    Give function list of all resized images for more efficient processing
    '''
    io.logger_setup()
    model = models.CellposeModel(gpu=True)
    masks, _, _ = model.eval(imgs, batch_size=32, z_axis=0, do_3D=False,normalize=True,stitch_threshold=0.6,cellprob_threshold=-0.5)
    return masks

def up_size_masks(masks, img):
    if masks.shape == img.shape:
        print('masks and image are the same size, no resizing applied')
        return masks
    else:
        return resize(masks,img.shape,preserve_range=True,order=0).astype(np.uint16)

def clear_xy_borders(mask_img,border=None):
    border == 1 if border == None else border == border
    bool_arr = np.zeros((mask_img.shape[0],mask_img.shape[1]-border*2,mask_img.shape[2]-border*2))
    xy_boarder = np.pad(bool_arr,pad_width=((0,0),(border,border),(border,border)),mode='constant',constant_values=1).astype(np.bool)
    labels, number = label(mask_img,background=0,return_num=True)
    indices = np.arange(number+1)
    border_indices = np.unique(labels[xy_boarder])
    label_mask = np.isin(indices, border_indices)
    mask = label_mask[labels.reshape(-1)].reshape(labels.shape)
    labels[mask] = 0
    return labels

def get_3D_surfacearea_and_sphericity(mask_img,pixel_spacing,props_df):
    objects = ndimage.find_objects(mask_img)
    measurements_dict = {}
    for i, sl in enumerate(objects):
        if sl is None:
            continue
        label_val = i + 1
        if label_val not in props_df['label']:
            continue
        # pad by 1 so surface is closed
        padded_sl = tuple(slice(max(0, s.start-1), s.stop+1) for s in sl)
        sub = mask_img[padded_sl]
        mask = (sub == label_val).astype(np.uint8)
        verts, faces, _, _ = marching_cubes(mask, level=0.5, spacing=pixel_spacing)
        surface_area = mesh_surface_area(verts, faces)
        vol = props_df.query(f'label=={label_val}')['area']
        sphericity = (np.pi**(1/3)*(6*vol)**(2/3))/surface_area
        measurements_dict[i] = {'label':label_val,
                                'surface_area_um2':surface_area,
                                'sphericity': sphericity}
    return measurements_dict

def get_shape_measurements(mask_img,pixel_spacing):
    props = ['label', 'bbox' , 'eccentricity', 'equivalent_diameter_area', 'extent', 'feret_diameter_max', 'solidity']
    props_dict = regionprops_table(mask_img,properties=props,spacing=pixel_spacing)
    props_df = pd.DataFrame.from_dict(props_dict)
    return props_df